# Figure 6 data: evaluation with appointment delays

- Loads the final Normalized, Fract, and CNN Mesh PPO checkpoints and the configured reference protocols.
- Evaluates them over 30 rounds with 10 parallel simulations and tumour IDs 1–10.
- Uses a fixed appointment delay distribution with 24h treatment availability.
- Saves data, summary statistics, and diagnostic plots; the summary is used by the Figure 6 comparison script.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

PROJECT_DIR = next((candidate.resolve() for parent in (Path.cwd(), *Path.cwd().parents) for candidate in (parent, parent / "report_files") if (candidate / "train_normalized_PPO.py").is_file() and (candidate / "TumorGrowthEnv/tumor_growth/__init__.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not locate report_files")
for local_dir in (PROJECT_DIR, PROJECT_DIR / "TumorGrowthEnv"):
    if str(local_dir) not in sys.path:
        sys.path.insert(0, str(local_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import gym
import tumor_growth
import cnn_mesh_policy

import pickle
import json

from stable_baselines3 import PPO

In [ ]:
GPU_ID = 0
ROUNDS = 30
PARALLEL_RUNS = 10
TUMOUR_IDS = list(range(1, 11))
AVAILABLE_HOURS = 24
OPENING_HOUR = 0
LAST_SELECTABLE_HOUR = OPENING_HOUR + AVAILABLE_HOURS - 1
MODE = "app_delays"
APPOINTMENT_DELAYS = True
RESULTS_DIR = PROJECT_DIR / "baselines/results"

MODEL_PATHS = {
    "ppo_normal": RESULTS_DIR / "models/normalized_ppo/normalized_ppo_20260904_202531_seed0/checkpoints/ppo_50000_steps.zip",
    "ppo_fract": RESULTS_DIR / "models/fract_ppo/fract_phase_mode_ox_ppo_20260904_202539_seed0/checkpoints/ppo_50000_steps.zip",
    "cnn_mesh_ppo": RESULTS_DIR / "models/cnn_mesh_ppo/cnn_mesh_ppo_20260920_152916_seed0/checkpoints/ppo_50000_steps.zip",
}
for mode, path in MODEL_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"{mode}: {path}")

### Full comparison with appointment delays

In [ ]:
class Baseline:

    env: gym.Env
    leftover_cells_history: dict
    valid_modes = [
        "ppo_normal", "ppo_fract", "cnn_mesh_ppo",
        "BMI", "BMII", "random", "no_treatment",
        "best_protocol", "popular_protocol", "two_fast_5gy_doses",
    ]
    mode_labels = {
        "paper_ppo": "Paper PPO",
        "ppo_normal": "Normalized PPO",
        "ppo_fract": "Fract PPO",
        "ppo_mesh": "Mesh PPO",
        "cnn_mesh_ppo": "CNN Mesh PPO",
        "best_protocol": "Best protocol",
        "popular_protocol": "Popular protocol",
        "BMI": "Benchmark I (BMI)",
        "BMII": "Benchmark II (BMII)",
        "random": "Random schedule",
        "no_treatment": "No treatment",
        "two_fast_5gy_doses": "5 Gy on days 1 and 2",
    }

    def __init__(self, gpu_id: int):
        self.gpu_id = gpu_id
        self.leftover_cells_history = {mode: [] for mode in self.valid_modes}
        self.run_records = {mode: [] for mode in self.valid_modes}
        self.step_records = []

    def get_model(self, mode: str):
        if mode not in MODEL_PATHS:
            return None
        model_path = MODEL_PATHS[mode]
        print(f"Loading {model_path}")
        return PPO.load(model_path, device="cpu")

    def get_environment_id(self, mode: str):
        return {
            "ppo_normal": "tumor_growth:tumor-growth-normalized-ppo-v0",
            "ppo_fract": "tumor_growth:tumor-growth-fract-ppo-v0",
            "cnn_mesh_ppo": "tumor_growth:tumor-growth-cnn-mesh-ppo-v0",
        }.get(mode, "tumor_growth:tumor-growth-normalized-ppo-v0")

    def _set_action(self, mode: str, step: int, obs=None):
        if mode == "no_treatment":
            return [0, 0]
        elif mode == "random":
            action = np.asarray(self.env.action_space.sample()).copy()
            return action
        elif mode == "BMI":
            return [6, 18, 5, 2]
        elif mode == "BMII":
            return [0, 4]
        elif mode == "best_protocol":
            if step == 0:
                return [23, 10]
            else:
                return [0, 10]
        elif mode == "popular_protocol":
            if step == 0:
                return [3, 10]
            else:
                return [10, 10]
        elif mode == "two_fast_5gy_doses":
            return [0, 10]
        raise ValueError(f"No baseline action is defined for {mode}")

    def constrain_action(self, requested_action):
        action = np.asarray(requested_action, dtype=np.int64).copy()
        hour_indices = (0, 1) if action.size == 4 else (0,)
        for index in hour_indices:
            clock_hour = np.clip(action[index], OPENING_HOUR, LAST_SELECTABLE_HOUR)
            action[index] = clock_hour - OPENING_HOUR
        return action

    def run(self, mode: str = "random", parallel_runs: int = 10, rounds: int = 10, steps_per_round: int = 5, with_time: bool = True, env_mode = None, appointment_delays: bool = False):
        if mode not in self.valid_modes:
            raise ValueError(f"Mode must be one of {self.valid_modes}")
            
        environment_id = self.get_environment_id(mode)
        print(f"Initializing {environment_id} for {mode}...")
        self.env = gym.make(
            environment_id,
            parallel_runs=parallel_runs,
            with_time=with_time,
            gpu_id=self.gpu_id,
            tumors_list=TUMOUR_IDS,
            appointment_delays=appointment_delays,
            dosing_hours=AVAILABLE_HOURS,
            **({"mode": env_mode} if env_mode is not None else {}),
        )

        model = self.get_model(mode)

        print("Simulation started successfully!\n")

        for round_idx in range(rounds):
            print(f"\n--- Round {round_idx + 1} ---")
            obs = self.env.reset()
            print(f"Starting with {self.env.get_num_cells()} cells")
            leftover_cells = 0
            actions = []
            requested_actions = []
            last_info = {}

            for step in range(steps_per_round):
                observation_before = np.asarray(obs).copy()
                model_observation = obs
                if model is None:
                    requested_action = self._set_action(mode, step, obs)
                else:
                    requested_action, _ = model.predict(model_observation, deterministic=True)
                action = self.constrain_action(requested_action)
                requested_actions.append(np.asarray(requested_action).tolist())
                actions.append(action.tolist())

                print(f"--- Step {step + 1} ---")
                print(f"Scheduled action: {action}")
                
                obs, reward, done, info = self.env.step(action)
                last_info = info
                self.step_records.append({
                    "mode": mode,
                    "round": round_idx + 1,
                    "step": step + 1,
                    "observation_size": int(observation_before.size),
                    "model_observation_size": int(np.asarray(model_observation).size),
                    "requested_action": np.asarray(requested_action).tolist(),
                    "applied_action": action.tolist(),
                    "scheduled_action": action.tolist(),
                    "reward": float(reward),
                    "done": bool(done),
                    "info": info,
                })

                
                if done:
                    leftover_cells = info.get("leftover_cells", 0)
                    print("Simulation reached an end state.")
                    break

            if last_info:
                leftover_cells = float(last_info.get("leftover_cells", np.nan))
                self.leftover_cells_history[mode].append(leftover_cells)
                self.run_records[mode].append({
                    "mode": mode,
                    "run": round_idx + 1,
                    "leftover_cells": leftover_cells,
                    "complete_kill_probability": float(last_info.get("killed_cancer_prob", np.nan)),
                    "cumulative_dose_gy": float(last_info.get("cumulative_dose", np.nan)),
                    "reward": float(reward),
                    "actions": actions,
                    "requested_actions": requested_actions,
                    "available_hours_per_day": AVAILABLE_HOURS,
                    "opening_hour": OPENING_HOUR,
                    "last_selectable_hour": LAST_SELECTABLE_HOUR,
                    "appointment_delays": appointment_delays,
                })
                print(f"Round {round_idx + 1} finished with {leftover_cells} leftover cells.")

        self.env.close()
        print("\nSimulation finished.")
        print(f"Leftover cells history for {mode} over {rounds} rounds: {self.leftover_cells_history[mode]}")

    def compare_baselines(self, parallel_runs: int = 10, rounds: int = 30, appointment_delays: bool = False):
        print("\nComparing Baselines:")
        self.step_records = []
        for mode in self.valid_modes:
            print(f"\n\nRunning baseline mode: {mode}")

            self.leftover_cells_history[mode] = []
            self.run_records[mode] = []

            if mode == "BMI":
                self.run(mode=mode, parallel_runs=parallel_runs, rounds=rounds, steps_per_round=4, env_mode="2doses", appointment_delays=appointment_delays)
            else:
                self.run(mode=mode, parallel_runs=parallel_runs, rounds=rounds, steps_per_round=5, appointment_delays=appointment_delays)
            
        print("\n\nBaseline Comparison Results (Leftover Cells per Round):\n")
        for mode in self.valid_modes:
            history = self.leftover_cells_history[mode]
            print(f"{mode}: {history}")
            mean = np.mean(history)
            print(f"mean: {mean}")

In [ ]:
if MODE != "app_delays":
    raise ValueError("This notebook configuration is for appointment-delay evaluation")
if not APPOINTMENT_DELAYS:
    raise ValueError("Appointment-delay mode requires APPOINTMENT_DELAYS=True")

baseline = Baseline(GPU_ID)
baseline.compare_baselines(
    parallel_runs=PARALLEL_RUNS,
    rounds=ROUNDS,
    appointment_delays=APPOINTMENT_DELAYS,
)
for mode in baseline.valid_modes:
    if len(baseline.run_records[mode]) != ROUNDS:
        raise RuntimeError(f"Expected {ROUNDS} completed runs for {mode}")


### Statistics

In [ ]:
from matplotlib.ticker import MultipleLocator
import pandas as pd
import seaborn as sns

mode_labels = Baseline.mode_labels
runs_df = pd.DataFrame([record for mode in baseline.valid_modes for record in baseline.run_records[mode]])
if runs_df.empty:
    raise RuntimeError("No evaluation runs were recorded")

summary = (
    runs_df.groupby("mode", sort=False)
    .agg(
        runs=("run", "size"),
        mean_leftover_cells=("leftover_cells", "mean"),
        median_leftover_cells=("leftover_cells", "median"),
        variance_leftover_cells=("leftover_cells", "var"),
        std_leftover_cells=("leftover_cells", "std"),
        complete_kill_probability=("complete_kill_probability", "mean"),
        mean_cumulative_dose_gy=("cumulative_dose_gy", "mean"),
    )
    .reset_index()
)
summary["variance_leftover_cells"] = summary["variance_leftover_cells"].fillna(0.0)
summary["std_leftover_cells"] = summary["std_leftover_cells"].fillna(0.0)
summary["protocol"] = summary["mode"].map(mode_labels)
summary = summary.sort_values("mean_leftover_cells").reset_index(drop=True)
runs_df["protocol"] = runs_df["mode"].map(mode_labels)
display(summary.round(3))

protocol_order = summary["protocol"].tolist()
fig_box, ax_box = plt.subplots(figsize=(15, 7))
sns.boxplot(x="protocol", y="leftover_cells", data=runs_df, order=protocol_order, color=sns.color_palette("crest", 3)[1], ax=ax_box)
ax_box.set_xlabel("")
ax_box.set_ylabel("Mean leftover cancer cells")
ax_box.set_axisbelow(True)
ax_box.yaxis.set_major_locator(MultipleLocator(10))
ax_box.grid(axis="y", which="major", linestyle="--", alpha=0.55)
ax_box.tick_params(axis="x", rotation=35)
fig_box.tight_layout()

fig_violin, ax_violin = plt.subplots(figsize=(15, 7))
sns.violinplot(x="protocol", y="leftover_cells", data=runs_df, order=protocol_order, color=sns.color_palette("crest", 3)[1], cut=0, inner="quart", ax=ax_violin)
ax_violin.set_title("PPO models and reference protocols with appointment delays")
ax_violin.set_xlabel("")
ax_violin.set_ylabel("Mean leftover cancer cells")
ax_violin.set_axisbelow(True)
ax_violin.yaxis.set_major_locator(MultipleLocator(10))
ax_violin.grid(axis="y", which="major", linestyle="--", alpha=0.55)
ax_violin.tick_params(axis="x", rotation=35)
fig_violin.tight_layout()

In [ ]:
results_dir = RESULTS_DIR
results_dir.mkdir(parents=True, exist_ok=True)
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
raw_export = runs_df.copy()
raw_export["actions"] = raw_export["actions"].map(json.dumps)
raw_export["requested_actions"] = raw_export["requested_actions"].map(json.dumps)
raw_path = results_dir / f"baseline_runs_{ROUNDS}x{PARALLEL_RUNS}_{AVAILABLE_HOURS}h_{MODE}_{timestamp}.csv"
summary_path = results_dir / f"baseline_summary_{ROUNDS}x{PARALLEL_RUNS}_{AVAILABLE_HOURS}h_{MODE}_{timestamp}.csv"
steps_path = results_dir / f"baseline_steps_{ROUNDS}x{PARALLEL_RUNS}_{AVAILABLE_HOURS}h_{MODE}_{timestamp}.jsonl"
boxplot_path = results_dir / f"baseline_boxplot_{ROUNDS}x{PARALLEL_RUNS}_{AVAILABLE_HOURS}h_{MODE}_{timestamp}.png"
violinplot_path = results_dir / f"baseline_violinplot_{ROUNDS}x{PARALLEL_RUNS}_{AVAILABLE_HOURS}h_{MODE}_{timestamp}.png"
raw_export.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)
with steps_path.open("w") as steps_file:
    for record in baseline.step_records:
        steps_file.write(json.dumps(record, default=lambda value: value.tolist() if hasattr(value, "tolist") else str(value)) + "\n")
fig_box.savefig(boxplot_path, dpi=300, bbox_inches="tight")
fig_violin.savefig(violinplot_path, dpi=300, bbox_inches="tight")
print(f"Saved runs to {raw_path}")
print(f"Saved summary to {summary_path}")
print(f"Saved compact step records to {steps_path}")
print(f"Saved boxplot to {boxplot_path}")
print(f"Saved violin plot to {violinplot_path}")